# Phase 2: Exploratory Data Analysis (EDA) & Complete Dataset Audit
### Project: Automated Classification of Martian Surface Images Captured by NASA's Curiosity Rover Using Machine Learning

This notebook provides interactive exploration of the NASA Curiosity MSL browse image dataset across all 6,691 split-referenced images:
- Complete dataset audit (resolving RGB vs Grayscale distribution, image formats, dimensions)
- Class distributions across official Train, Validation, and Test splits
- Class imbalance analysis (dataset-wide and per-split metrics)
- Visual exploration via representative class contact sheet

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure project root is on sys.path
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import config
print(f"Project Root: {config.PROJECT_ROOT}")
print(f"Dataset Directory: {config.DATA_DIR}")

## 1. Class Distribution Across Official NASA Splits
Load official class mapping and split files to view the complete class breakdown.

In [ ]:
# Load class mapping
class_names = {}
with open(config.CLASS_MAPPING_PATH, "r") as f:
    for line in f:
        parts = line.strip().split(None, 1)
        if len(parts) == 2 and parts[0].isdigit():
            class_names[int(parts[0])] = parts[1].strip()

# Count samples per split
def count_split(filepath):
    counts = {cid: 0 for cid in config.CLASS_IDS_RANGE}
    with open(filepath) as f:
        for line in f:
            if line.strip():
                lbl = int(line.strip().split()[1])
                counts[lbl] += 1
    return counts

train_counts = count_split(config.TRAIN_LABELS_PATH)
val_counts = count_split(config.VAL_LABELS_PATH)
test_counts = count_split(config.TEST_LABELS_PATH)

df_summary = pd.DataFrame([
    {
        "Class ID": cid,
        "Class Name": class_names[cid],
        "Train": train_counts[cid],
        "Validation": val_counts[cid],
        "Test": test_counts[cid],
        "Total": train_counts[cid] + val_counts[cid] + test_counts[cid],
        "Percentage (%)": round((train_counts[cid] + val_counts[cid] + test_counts[cid]) / 6691 * 100, 2)
    }
    for cid in config.CLASS_IDS_RANGE
])

print(f"Total Images: {df_summary[Total].sum()}")
df_summary

## 2. Key Audit Findings
1. **Total referenced images**: 6,691 (3,746 Train, 1,640 Val, 1,305 Test).
2. **Color Mode Resolution**: 6,519 RGB (97.43%) and 172 Grayscale Mode L (2.57%). 0 Corrupted.
3. **Dimensions**: All widths are 255 or 256 px. Heights vary from 85 to 279 px.
4. **Class Imbalance**: Largest class is `ground` (2,684 images / 40.11%); Smallest active class is `portion tube opening` (22 images / 0.33%). Imbalance ratio is **122.00x**.
5. **Zero-Instance Class**: Class 22 (`sun`) has 0 samples across all splits.

In [ ]:
# Display generated EDA plots
from IPython.display import Image as IPyImage, display

display(IPyImage(filename=str(config.PLOTS_DIR / "01_overall_class_distribution.png")))
display(IPyImage(filename=str(config.PLOTS_DIR / "02_split_class_distributions.png")))
display(IPyImage(filename=str(config.PLOTS_DIR / "03_image_dimension_distribution.png")))
display(IPyImage(filename=str(config.PLOTS_DIR / "04_channel_color_mode_distribution.png")))
display(IPyImage(filename=str(config.PLOTS_DIR / "05_representative_classes_contact_sheet.png")))